In [1]:
# Se importan las librerías necesarias.

import ast
import matplotlib.pyplot as plt
import pandas as pd
import xgboost as xgb

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay, roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MultiLabelBinarizer
from skopt import BayesSearchCV

In [2]:
# Se carga el conjunto de datos para el entrenamiento del modelo
trainingData = pd.read_csv('raw_ZumbaConsumerAppChurn30D.csv')
trainingData['NEW_PAID_SUBSCRIPTION_DATE'] = pd.to_datetime(trainingData['NEW_PAID_SUBSCRIPTION_DATE'])



# Se filtran las suscripciones anuales. Dado que estas tienen una duración alta, es una varaible que puede sesga el análisis del Churn a 30 días.
trainingData = trainingData[trainingData['SUBSCRIPTION_TIER'] != 'Annual']

# Se eliminan columnas que no aportan información relevanta y aquellas que generan autocorrelación con otras columnas.
trainingData = trainingData.drop(columns = ['USER_ID', 'SUBSCRIPTION_ID', 'NEW_PAID_SUBSCRIPTION_DATE', 'PAID_SUBSCRIPTION_DURATION_DAYS', 'REGION_TYPE_LEVEL_1', 'REGION_TYPE_LEVEL_2', 'SUBSCRIPTION_AFFILIATE', 'MOTIVATING_FACTORS', 'CLASS_FORMAT_PREFERENCE', 'GENDER_IDENTITY', 'AGE'])

# Se eliminan filas con valores nulos en columnas categóricas relevantes, dado que el número de nulos de estas columnas es bajo.
trainingData = trainingData.dropna(subset = ['DEVICE_BRAND', 'DEVICE_OPERATING_SYSTEM', 'CLASS_INTENSITY_PREFERENCE', 'DANCE_LEVEL_PREFERENCE'])



# Las columnas de tipo 'lista' que tienen valores nulos son imputadas con listas vacías.
trainingData['CLASS_INTENSITY_PREFERENCE'] = trainingData['CLASS_INTENSITY_PREFERENCE'].fillna('[]')
trainingData['MUSIC_PREFERENCE'] = trainingData['MUSIC_PREFERENCE'].fillna('[]')

# A fin de implementar XGBoost de manera correcta, la columna FITNESS_GOAL_PREFERENCE es imputada con la categoría 'Unknown'.
trainingData['FITNESS_GOAL_PREFERENCE'] = trainingData['FITNESS_GOAL_PREFERENCE'].fillna('Unknown')
# Se definen las variables de tipo 'lista' que serán convertidas en variables dummy.
listColumns = ['CLASS_INTENSITY_PREFERENCE', 'MUSIC_PREFERENCE']

# Se convierten las columnas de tipo 'lista' en variables Dummy.
for col in listColumns:
    trainingData[col] = trainingData[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

mlb = MultiLabelBinarizer()
for col in listColumns:
    dummies = pd.DataFrame(mlb.fit_transform(trainingData[col]), columns = [f'{col}_{cl}' for cl in mlb.classes_], index = trainingData.index)
    trainingData = pd.concat([trainingData, dummies], axis = 1)

# Se eliminan las columnas originales de tipo 'lista'.
trainingData = trainingData.drop(columns = listColumns)



# A fin de utilizar XGBoost, se convierten las columnas categóricas al tipo 'category'.

categoricalColumns = [
    'SUBSCRIPTION_TIER',
    'SUBSCRIPTION_SOURCE',
    'SUBSCRIPTION_PROVIDER',
    'COUNTRY',
    'DANCE_LEVEL_PREFERENCE',
    'FITNESS_GOAL_PREFERENCE',
    'APP_VERSION',
    'DEVICE_OPERATING_SYSTEM',
    'DEVICE_MANUFACTURER',
    'DEVICE_BRAND',
    'DEVICE_MODEL'
    ]

trainingData[categoricalColumns] = trainingData[categoricalColumns].astype('category')



# Mejores variables para el modelo XGBoost tras un modelo de selección 'Backward Feature Elimination'.
variablesModelo = ['TRIAL_DURATION_DAYS', 'SUBSCRIPTION_TIER', 'SUBSCRIPTION_SOURCE', 'SUBSCRIPTION_PROVIDER', 'COUNTRY', 'CLASS_PER_WEEK_GOAL', 'DANCE_LEVEL_PREFERENCE', 'FITNESS_GOAL_PREFERENCE', 'APP_VERSION', 'DEVICE_OPERATING_SYSTEM', 'DEVICE_BRAND', 'APP_INSTALL_TO_PAID_SUBSCRIPTION_DAYS', 'NO_VIDEO_STARTED_30D', 'NO_VIDEO_WATCHED_30D', 'AVG_VIDEO_WATCHED_PERCENTAGE_30D', 'AVG_VIDEO_LENGTH_MIN_30D', 'CLASS_INTENSITY_PREFERENCE_High', 'CLASS_INTENSITY_PREFERENCE_Low', 'CLASS_INTENSITY_PREFERENCE_Medium', 'MUSIC_PREFERENCE_Afro Rhythms', 'MUSIC_PREFERENCE_Afrobeats', 'MUSIC_PREFERENCE_Alternative', 'MUSIC_PREFERENCE_Bachata', 'MUSIC_PREFERENCE_Belly Dance', 'MUSIC_PREFERENCE_Bellydance', 'MUSIC_PREFERENCE_Bhangra', 'MUSIC_PREFERENCE_Blues', 'MUSIC_PREFERENCE_Bollywood', 'MUSIC_PREFERENCE_Brazilian Rhythms', 'MUSIC_PREFERENCE_Broadway', 'MUSIC_PREFERENCE_Caribbean Rhythms', 'MUSIC_PREFERENCE_Chill Out', 'MUSIC_PREFERENCE_Country', 'MUSIC_PREFERENCE_Cumbia', 'MUSIC_PREFERENCE_Disco', 'MUSIC_PREFERENCE_Electronic', 'MUSIC_PREFERENCE_House', 'MUSIC_PREFERENCE_K-Pop', 'MUSIC_PREFERENCE_Merengue', 'MUSIC_PREFERENCE_Other', 'MUSIC_PREFERENCE_Pop', 'MUSIC_PREFERENCE_R&B', 'MUSIC_PREFERENCE_Reggae', 'MUSIC_PREFERENCE_Reggaeton', 'MUSIC_PREFERENCE_Rock', 'MUSIC_PREFERENCE_Salsa', 'MUSIC_PREFERENCE_Soca', 'MUSIC_PREFERENCE_Techno', 'MUSIC_PREFERENCE_World Rhythms']

# Se dividen los datos entre las variables independientes y dependiente.
xTraining = trainingData[variablesModelo]
yTraining = trainingData['PAID_SUBSCRIPTION_CHURN_30D']

In [3]:
# Se carga el conjunto de datos sobre los cuales se realizan las predicciones de abandono.
predictData = pd.read_csv('raw_NewPredictChurnData.csv')
predictData['NEW_PAID_SUBSCRIPTION_DATE'] = pd.to_datetime(predictData['NEW_PAID_SUBSCRIPTION_DATE'])



# Se filtran las suscripciones anuales. Dado que estas tienen una duración alta, es una varaible que puede sesga el análisis del Churn a 30 días.
predictData = predictData[predictData['SUBSCRIPTION_TIER'] != 'Annual']

# Se eliminan columnas que no aportan información relevanta y aquellas que generan autocorrelación con otras columnas.
predictData = predictData.drop(columns = ['USER_ID', 'SUBSCRIPTION_ID', 'NEW_PAID_SUBSCRIPTION_DATE', 'PAID_SUBSCRIPTION_DURATION_DAYS', 'REGION_TYPE_LEVEL_1', 'REGION_TYPE_LEVEL_2', 'SUBSCRIPTION_AFFILIATE', 'MOTIVATING_FACTORS', 'CLASS_FORMAT_PREFERENCE', 'GENDER_IDENTITY', 'AGE'])

# Se eliminan filas con valores nulos en columnas categóricas relevantes, dado que el número de nulos de estas columnas es bajo.
predictData = predictData.dropna(subset = ['DEVICE_BRAND', 'DEVICE_OPERATING_SYSTEM', 'CLASS_INTENSITY_PREFERENCE', 'DANCE_LEVEL_PREFERENCE'])



# Las columnas de tipo 'lista' que tienen valores nulos son imputadas con listas vacías.
predictData['CLASS_INTENSITY_PREFERENCE'] = predictData['CLASS_INTENSITY_PREFERENCE'].fillna('[]')
predictData['MUSIC_PREFERENCE'] = predictData['MUSIC_PREFERENCE'].fillna('[]')

# A fin de implementar XGBoost de manera correcta, la columna FITNESS_GOAL_PREFERENCE es imputada con la categoría 'Unknown'.
predictData['FITNESS_GOAL_PREFERENCE'] = predictData['FITNESS_GOAL_PREFERENCE'].fillna('Unknown')
# Se definen las variables de tipo 'lista' que serán convertidas en variables dummy.
listColumns = ['CLASS_INTENSITY_PREFERENCE', 'MUSIC_PREFERENCE']

# Se convierten las columnas de tipo 'lista' en variables Dummy.
for col in listColumns:
    predictData[col] = predictData[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

mlb = MultiLabelBinarizer()
for col in listColumns:
    dummies = pd.DataFrame(mlb.fit_transform(predictData[col]), columns = [f'{col}_{cl}' for cl in mlb.classes_], index = predictData.index)
    predictData = pd.concat([predictData, dummies], axis = 1)

# Se eliminan las columnas originales de tipo 'lista'.
predictData = predictData.drop(columns = listColumns)



# A fin de utilizar XGBoost, se convierten las columnas categóricas al tipo 'category'.

categoricalColumns = [
    'SUBSCRIPTION_TIER',
    'SUBSCRIPTION_SOURCE',
    'SUBSCRIPTION_PROVIDER',
    'COUNTRY',
    'DANCE_LEVEL_PREFERENCE',
    'FITNESS_GOAL_PREFERENCE',
    'APP_VERSION',
    'DEVICE_OPERATING_SYSTEM',
    'DEVICE_MANUFACTURER',
    'DEVICE_BRAND',
    'DEVICE_MODEL'
    ]

predictData[categoricalColumns] = predictData[categoricalColumns].astype('category')



# Mejores variables para el modelo XGBoost tras un modelo de selección 'Backward Feature Elimination'.
variablesModelo = ['TRIAL_DURATION_DAYS', 'SUBSCRIPTION_TIER', 'SUBSCRIPTION_SOURCE', 'SUBSCRIPTION_PROVIDER', 'COUNTRY', 'CLASS_PER_WEEK_GOAL', 'DANCE_LEVEL_PREFERENCE', 'FITNESS_GOAL_PREFERENCE', 'APP_VERSION', 'DEVICE_OPERATING_SYSTEM', 'DEVICE_BRAND', 'APP_INSTALL_TO_PAID_SUBSCRIPTION_DAYS', 'NO_VIDEO_STARTED_30D', 'NO_VIDEO_WATCHED_30D', 'AVG_VIDEO_WATCHED_PERCENTAGE_30D', 'AVG_VIDEO_LENGTH_MIN_30D', 'CLASS_INTENSITY_PREFERENCE_High', 'CLASS_INTENSITY_PREFERENCE_Low', 'CLASS_INTENSITY_PREFERENCE_Medium', 'MUSIC_PREFERENCE_Afro Rhythms', 'MUSIC_PREFERENCE_Afrobeats', 'MUSIC_PREFERENCE_Alternative', 'MUSIC_PREFERENCE_Bachata', 'MUSIC_PREFERENCE_Belly Dance', 'MUSIC_PREFERENCE_Bellydance', 'MUSIC_PREFERENCE_Bhangra', 'MUSIC_PREFERENCE_Blues', 'MUSIC_PREFERENCE_Bollywood', 'MUSIC_PREFERENCE_Brazilian Rhythms', 'MUSIC_PREFERENCE_Broadway', 'MUSIC_PREFERENCE_Caribbean Rhythms', 'MUSIC_PREFERENCE_Chill Out', 'MUSIC_PREFERENCE_Country', 'MUSIC_PREFERENCE_Cumbia', 'MUSIC_PREFERENCE_Disco', 'MUSIC_PREFERENCE_Electronic', 'MUSIC_PREFERENCE_House', 'MUSIC_PREFERENCE_K-Pop', 'MUSIC_PREFERENCE_Merengue', 'MUSIC_PREFERENCE_Other', 'MUSIC_PREFERENCE_Pop', 'MUSIC_PREFERENCE_R&B', 'MUSIC_PREFERENCE_Reggae', 'MUSIC_PREFERENCE_Reggaeton', 'MUSIC_PREFERENCE_Rock', 'MUSIC_PREFERENCE_Salsa', 'MUSIC_PREFERENCE_Soca', 'MUSIC_PREFERENCE_Techno', 'MUSIC_PREFERENCE_World Rhythms']

# Se definen las variables independientes.
xPredict = predictData[variablesModelo]

In [4]:
# Se entrena el modelo XGBoost usando el conjunto de datos de entrenamiento, usando los mejores hiperparámetros encontrados.
xgbModel = xgb.XGBClassifier(
    colsample_bytree = 0.6, 
    learning_rate = 0.035, 
    max_depth = 5, 
    min_child_weight = 9, 
    n_estimators = 300, 
    reg_lambda = 8, 
    subsample = 0.9,  
    enable_categorical = True,
    eval_metric = 'auc',
    random_state = 0
)
xgbModel.fit(xTraining, yTraining)

# Se generan las predicciones del modelo.
yPred = xgbModel.predict(xPredict)
yPredProb = xgbModel.predict_proba(xPredict)[:, 1]

In [5]:
predictData['PREDICTED_PROBABILITY'] = yPredProb
predictData['PREDICTED_CLASS'] = yPred

predictData = predictData.drop(columns = ['PAID_SUBSCRIPTION_CHURN_30D'])
predictData.to_csv('predictions_ChurnPredictedData.csv')
predictData.head(10)

,TRIAL_DURATION_DAYS,SUBSCRIPTION_TIER,SUBSCRIPTION_SOURCE,SUBSCRIPTION_PROVIDER,COUNTRY,CLASS_PER_WEEK_GOAL,DANCE_LEVEL_PREFERENCE,FITNESS_GOAL_PREFERENCE,APP_VERSION,DEVICE_OPERATING_SYSTEM,...,MUSIC_PREFERENCE_R&B,MUSIC_PREFERENCE_Reggae,MUSIC_PREFERENCE_Reggaeton,MUSIC_PREFERENCE_Rock,MUSIC_PREFERENCE_Salsa,MUSIC_PREFERENCE_Soca,MUSIC_PREFERENCE_Techno,MUSIC_PREFERENCE_World Rhythms,PREDICTED_PROBABILITY,PREDICTED_CLASS
1,0,Monthly,App,Google,Croatia,5.0,Intermediate,lose weight and tone,3.14.0,Android,...,0,0,1,1,1,0,0,0,0.523972,1
3,0,Monthly,App,Apple,Thailand,5.0,Advanced,lose weight and tone,2,iOS,...,0,0,1,0,0,0,1,0,0.641809,1
12,0,Monthly,Web,App to Web,United States of America,3.0,Beginner,Have some fun,6,iOS,...,0,0,0,0,0,0,0,0,0.530659,1
15,0,Monthly,Web,App to Web,United States of America,3.0,Beginner,Unknown,6,iOS,...,0,0,1,0,1,0,0,0,0.482926,0
17,0,Monthly,Web,Stripe,Spain,5.0,Advanced,lose weight and tone,3.13.1,Android,...,0,0,1,0,1,0,0,0,0.488660,0
19,7,Monthly,App,Apple,Netherlands,3.0,Advanced,lose weight and tone,3.14.0,Android,...,1,0,1,0,1,0,0,0,0.236268,0
22,0,Monthly,Web,App to Web,United States of America,4.0,Intermediate,lose weight and tone,6,iOS,...,0,0,1,0,1,0,0,0,0.422201,0
23,0,Monthly,App,App to Web,United States of America,3.0,Intermediate,lose weight and tone,6,iOS,...,0,0,1,0,1,0,1,1,0.366757,0
26,0,Monthly,App,Apple,Finland,3.0,Beginner,lose weight and tone,2,iOS,...,0,0,0,0,1,0,0,1,0.543285,1
28,0,Monthly,App,Google,Germany,2.0,Beginner,lose weight and tone,3.13.1,Android,...,0,0,0,0,0,0,0,0,0.728101,1
